In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Sweep the delay of $y[n] = x[n] + g \cdot y[n - M]$ from half a millisecond
to a second and a half. The strip at the top marks which effect we hear. Below,
the impulse response (left) spreads out in time as the comb (right) packs its
teeth together. When the echoes are too dense to draw one by one, the
impulse response is shaded instead. The audio card plays a guitar arpeggio
through the delay line.

In [ ]:
# hide
# autorun
MS0, G0 = 1000 * 200 / 44100, 0.6   # M = 200 samples, as in the comb-filtering example

guitar = pq.Audio.from_file("../assets/guitar-arpeggio.wav").as_mono()
SR = guitar.sample_rate
X = np.asarray(guitar.samples).reshape(-1)
X = np.concatenate([X / np.abs(X).max(), np.zeros(3 * SR)])   # room for the tail
T_IR, F_MAX = 2.0, 1000.0
COMB_MS, ECHO_MS = 50.0, 1000.0     # f_s / M = 20 Hz, and "on the order of seconds"


def stems(n, v):
    """x and y arrays that draw stems from 0 to v at each n, as one trace."""
    xs = np.repeat(np.asarray(n, float), 3)
    ys = np.column_stack([np.zeros(len(v)), v, np.full(len(v), np.nan)]).ravel()
    xs[2::3] = np.nan
    return xs, ys


def impulse_response(M, g, stems=stems, SR=SR, T_IR=T_IR, MAX_STEMS=500):
    """The impulse response as stems, or, when the echoes are too dense to
    draw one by one, the decaying envelope they fill (shaded)."""
    k = np.arange(int(T_IR * SR) // M + 1)
    if len(k) <= MAX_STEMS:
        return (*stems(k * M / SR, g ** k), "none")
    t = np.linspace(0.0, T_IR, 600)
    return t, g ** (t * SR / M), "tozeroy"


def response_db(M, g, SR=SR, F_MAX=F_MAX):
    teeth = np.arange(0.0, F_MAX, SR / M)
    f = np.union1d(np.linspace(0.0, F_MAX, 3000), teeth)
    return f, -20 * np.log10(np.abs(1 - g * np.exp(-2j * np.pi * f * M / SR)))


def feedback_delay(x, M, g):
    """y[n] = x[n] + g * y[n - M]. Unrolling one step of the feedback gives
    y[n] = (x[n] + g x[n - M]) + g^2 y[n - 2M], the same filter with twice the
    delay. We unroll until the delay is long, then compute it one block of
    D samples at a time: each block depends only on the block before it."""
    y, D, G = x.copy(), M, g
    while D < 4096 and G > 1e-9:
        y[D:] = y[D:] + G * y[:-D]       # the right side is evaluated first
        D, G = 2 * D, G * G
    for a in range(D, len(y), D):
        b = min(a + D, len(y))
        y[a:b] += G * y[a - D:b - D]
    return y


def figure():
    M = round(MS0 / 1000 * SR)
    fig = make_subplots(rows=2, cols=2, row_heights=[0.2, 0.8],
                        specs=[[{"colspan": 2}, None], [{}, {}]],
                        vertical_spacing=0.2, horizontal_spacing=0.12)
    for lo, hi, color, label in [(0.5, COMB_MS, "rgba(196,18,48,0.12)", "comb filtering"),
                                 (COMB_MS, ECHO_MS, "rgba(0,123,192,0.12)", "delay"),
                                 (ECHO_MS, 1500, "rgba(253,181,21,0.25)", "echo")]:
        fig.add_scatter(x=[lo, lo, hi, hi, lo], y=[0, 1, 1, 0, 0], fill="toself",
                        fillcolor=color, line=dict(width=0), hoverinfo="skip",
                        row=1, col=1)
        fig.add_annotation(x=np.log10(np.sqrt(lo * hi)), y=0.5, text=label,
                           showarrow=False, font=dict(size=12), xref="x", yref="y")
    fig.add_scatter(x=[MS0, MS0], y=[0, 1], mode="lines",
                    line=dict(color=IRON, width=3), row=1, col=1)
    xs, ys, fill = impulse_response(M, G0)
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color=RED, width=1.2),
                    fill=fill, fillcolor="rgba(196,18,48,0.3)", row=2, col=1)
    f, db = response_db(M, G0)
    fig.add_scatter(x=f, y=db, mode="lines", line=dict(color=RED, width=1.4),
                    row=2, col=2)
    fig.update_xaxes(type="log", range=[np.log10(0.5), np.log10(1500)],
                     title_text="Delay M / fₛ (ms)", fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[0, 1], showticklabels=False, ticks="", showline=False,
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, T_IR], title_text="Time (s)", fixedrange=True,
                     row=2, col=1)
    fig.update_yaxes(range=[0, 1.08], title_text="h[n]", fixedrange=True,
                     row=2, col=1)
    fig.update_xaxes(range=[0, F_MAX], title_text="Frequency (Hz)",
                     fixedrange=True, row=2, col=2)
    fig.update_yaxes(range=[-10, 28], title_text="Magnitude (dB)", fixedrange=True,
                     row=2, col=2)
    fig.update_layout(height=430)
    return fig


def controls(fig):
    ms = widgets.FloatLogSlider(description="Delay $M / f_s$ (ms)", base=10,
                                min=np.log10(MS0) - 0.96, max=np.log10(1500),
                                step=0.005, value=MS0, readout_format=".1f")
    g = widgets.FloatSlider(description="Gain $g$", min=0.0, max=0.95,
                            step=0.01, value=G0)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(ms, g, impulse_response=impulse_response,
               response_db=response_db, SR=SR, readout=readout):
        M = max(1, round(ms / 1000 * SR))
        xs, ys, fill = impulse_response(M, g)
        f, db = response_db(M, g)
        with fig.batch_update():
            fig.data[3].x = [ms, ms]
            fig.data[4].x, fig.data[4].y = xs, ys
            fig.data[4].fill = fill
            fig.data[5].x, fig.data[5].y = f, db
        rate = SR / M
        if ms < COMB_MS:
            kind = (f"above 20 Hz, so we hear the teeth as a pitch: comb "
                    f"filtering at {rate:.1f} Hz")
        elif ms < ECHO_MS:
            kind = (f"below 20 Hz, so the copies separate into a delay of "
                    f"{rate:.1f} repeats per second")
        else:
            kind = ("so late that each copy sounds like a reflection from a "
                    "distant surface: an echo")
        readout.value = (f"<span style='font-size:0.9em'><i>M</i> = {M:,} samples"
                         f" &nbsp;·&nbsp; <i>f</i><sub>s</sub> / <i>M</i> = "
                         f"{rate:,.1f} Hz, {kind}</span>")

    widgets.interactive_output(update, {"ms": ms, "g": g})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(X=X, feedback_delay=feedback_delay, SR=SR):
        M = max(1, round(ms.value / 1000 * SR))
        y = feedback_delay(X, M, g.value)
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (ms, g):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([ms, g, readout, out, gate])

icm_plotly.show(figure, controls)